# **Day 27: Recursion**

**Recursion** means a function **calling itself**.

Think of standing between two mirrors: you see a mirror inside a mirror inside a mirror... But in programming, it must **stop** at some point.

Every recursive function needs two parts:
1. **Base case:** The simple situation where the function stops.
2. **Recursive case:** The function calls itself with a **smaller** problem.

In [1]:
def countdown(n):
    if n == 0:              # Base case
        print("Go!")
        return
    print(n)
    countdown(n - 1)        # Recursive case (smaller problem)

countdown(3)

3
2
1
Go!


## **Factorial**
5! = 5 x 4 x 3 x 2 x 1. We can also say: **5! = 5 x 4!**. That is recursion.

In [2]:
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))
print(factorial(20))

120
2432902008176640000


How it works for `factorial(4)`:
```
factorial(4) = 4 * factorial(3)
                   = 3 * factorial(2)
                         = 2 * factorial(1)
                               = 1
Result = 4 * 3 * 2 * 1 = 24
```

## **Fibonacci (and Why Plain Recursion Can Be Slow)**

In [3]:
calls = 0

def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(25), "- number of function calls:", calls)

75025 - number of function calls: 242785


So many calls happen because the same values are calculated again and again. We can **remember** old answers using `lru_cache`. This idea is called **memoization**.

In [4]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

print(fib_fast(100))

354224848179261915075


## **Recursion Limit**
Python stops recursion after about 1000 levels, to protect the computer's memory.

In [5]:
import sys
print(sys.getrecursionlimit())

# def forever(n):
#     return forever(n + 1)
# forever(0)          # This will raise a RecursionError (no base case)

1000


## **Example from GIS: Flood Fill (Finding a Connected Water Body)**
In a satellite image, a **lake** is a group of water pixels that **touch** each other. If we start from one water pixel, we can find the whole lake by visiting its neighbours, then their neighbours, and so on.

This is called **flood fill** (the same idea as the "paint bucket" tool in MS Paint). It is very naturally written with recursion.

In the grid below: `1` = water, `0` = land.

In [6]:
grid = [
    [0, 0, 1, 1, 0, 0, 0],
    [0, 1, 1, 1, 0, 0, 1],
    [0, 0, 1, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0, 0, 0],
]

def flood_fill(r, c, visited):
    # Base cases: outside the grid, land, or already visited
    if r < 0 or r >= len(grid) or c < 0 or c >= len(grid[0]):
        return 0
    if grid[r][c] == 0 or (r, c) in visited:
        return 0
    visited.add((r, c))
    # Recursive case: this pixel + all connected pixels (up, down, left, right)
    return (1 + flood_fill(r - 1, c, visited) + flood_fill(r + 1, c, visited)
              + flood_fill(r, c - 1, visited) + flood_fill(r, c + 1, visited))

visited = set()
lakes = []
for r in range(len(grid)):
    for c in range(len(grid[0])):
        if grid[r][c] == 1 and (r, c) not in visited:
            size = flood_fill(r, c, visited)
            lakes.append(size)

print("Number of water bodies:", len(lakes))
print("Size of each (in pixels):", lakes)
print("Area of each (Sentinel-2, 10 m pixels):", [s * 100 for s in lakes], "square metre")

Number of water bodies: 3
Size of each (in pixels): [6, 4, 2]
Area of each (Sentinel-2, 10 m pixels): [600, 400, 200] square metre


Let us understand the code above:
- `visited` is a set that remembers pixels we have already counted, so we never count a pixel twice.
- For each water pixel, the function adds 1 and then calls itself for the four neighbours.
- The outer loops start a new flood fill whenever they find a water pixel that is not visited yet. Each new start means a **new, separate** water body.

## **More Examples**

In [7]:
def sum_digits(n):
    return n if n < 10 else n % 10 + sum_digits(n // 10)
print(sum_digits(700091))

def reverse(s):
    return s if len(s) <= 1 else reverse(s[1:]) + s[0]
print(reverse("Siliguri"))

17
irugiliS


In [8]:
# Flatten a nested list
nested = ["Kolkata", ["Howrah", ["Bally", "Liluah"], "Shibpur"], [["Salt Lake"]]]

def flatten(lst):
    out = []
    for x in lst:
        if isinstance(x, list):
            out.extend(flatten(x))
        else:
            out.append(x)
    return out

print(flatten(nested))

['Kolkata', 'Howrah', 'Bally', 'Liluah', 'Shibpur', 'Salt Lake']


In [9]:
# Tower of Hanoi
def hanoi(n, src="A", aux="B", dst="C"):
    if n == 0:
        return 0
    moves = hanoi(n - 1, src, dst, aux)
    print(f"Move disk {n} from {src} to {dst}")
    moves += 1
    moves += hanoi(n - 1, aux, src, dst)
    return moves

print("Total moves:", hanoi(3))

Move disk 1 from A to C
Move disk 2 from A to B
Move disk 1 from C to B
Move disk 3 from A to C
Move disk 1 from B to A
Move disk 2 from B to C
Move disk 1 from A to C
Total moves: 7


In [10]:
# Quick question
# Guess the output of this code cell
# def f(n):
#     if n == 0:
#         return 0
#     return n + f(n - 1)
# print(f(4))